# 📓 Exercise 01 — TF-IDF: Term Frequency & Inverse Document Frequency

**Series:** RAG Foundations | **Difficulty:** ⭐ Beginner  
**Time to complete:** ~30 minutes

---

## 🎯 Learning Objectives
By the end of this notebook you will be able to:
1. Explain intuitively what TF and IDF each measure
2. Calculate TF-IDF scores manually for a small corpus
3. Use `scikit-learn`'s `TfidfVectorizer` for production-style retrieval
4. Visualise and interpret TF-IDF score heatmaps
5. Complete fill-in-the-blank exercises to reinforce understanding

---

## 📖 Concept: Why TF-IDF?

When we search for a document using keywords, not all word matches are equally useful.

**The problem with simple word counting:**
- The word `"the"` appears in *every* document — it tells us nothing about relevance.
- The word `"chandrayaan"` appears in only *one* document — it's highly distinctive.

**TF-IDF solves this by combining two signals:**

| Signal | Full Name | Intuition |
|--------|-----------|----------|
| **TF** | Term Frequency | How often does this word appear *in this document*? |
| **IDF** | Inverse Document Frequency | How *rare* is this word across the entire collection? |

```
TF(word, doc)   = count(word in doc) / total words in doc
IDF(word)       = log( (N+1) / (docs containing word + 1) ) + 1  [smoothed]
TF-IDF(word,doc) = TF × IDF
```

> **Key insight:** A word that appears often in ONE document but rarely in OTHERS gets a HIGH score.
> A word that appears in ALL documents (like "the") gets a LOW score.

---

## ⚙️ Setup

In [ ]:
# Install required packages (run once)
!pip install numpy scikit-learn pandas matplotlib seaborn --quiet

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ All libraries loaded successfully!")

---
## 🔬 Part 1: Understanding Term Frequency (TF)

### What is TF?
Term Frequency answers: **"How important is this word to THIS document?"**

Formula:
```
TF(word, doc) = (number of times word appears in doc) / (total words in doc)
```

The division by document length ensures that *longer documents don't automatically get higher scores*.

**Example:**
- Doc: `"the moon is bright the moon is round"` (8 words)
- TF("moon") = 2/8 = **0.25**
- TF("the")  = 2/8 = **0.25**  ← same score! IDF will fix this.

In [ ]:
# ── DEMO: Calculating TF manually ──

def compute_tf(word, doc_tokens):
    """
    Compute Term Frequency of a word in a document.
    
    Parameters:
        word       : str  — the word to score
        doc_tokens : list — the document split into tokens/words
    
    Returns:
        float — TF score (0.0 to 1.0)
    """
    count = sum(1 for token in doc_tokens if token == word)
    return count / len(doc_tokens)

# Sample documents
doc1 = "the moon is bright the moon is round".split()
doc2 = "chandrayaan explored the moon cheaply".split()
doc3 = "the cat sat on the mat".split()

# Calculate TF for several words
words_to_check = ["moon", "the", "chandrayaan", "bright"]

print("TF Scores:")
print(f"{'Word':<15} {'Doc 1':>10} {'Doc 2':>10} {'Doc 3':>10}")
print("-" * 50)
for word in words_to_check:
    tf1 = compute_tf(word, doc1)
    tf2 = compute_tf(word, doc2)
    tf3 = compute_tf(word, doc3)
    print(f"  {word:<13} {tf1:>10.4f} {tf2:>10.4f} {tf3:>10.4f}")

print("\n📝 Notice: 'moon' has TF=0 in Doc3 (not present), 'chandrayaan' only exists in Doc2.")

### ✏️ Exercise 1.1 — Fill in the TF calculation

Given the document: `"india india launched moon mission moon"`

Fill in the blanks:
- Total word count = **???**
- TF("india") = **???** / **???** = **???**
- TF("moon")  = **???** / **???** = **???**
- TF("rocket") = **???** / **???** = **???**

In [ ]:
# ✏️ YOUR TURN: Complete this exercise
exercise_doc = "india india launched moon mission moon".split()

# TODO: Fill in these values
total_words = len(exercise_doc)    # Count the words
tf_india  = compute_tf("india",   exercise_doc)   # Calculate TF
tf_moon   = compute_tf("moon",    exercise_doc)
tf_rocket = compute_tf("rocket",  exercise_doc)

print("Exercise 1.1 Results:")
print(f"  Document       : {' '.join(exercise_doc)}")
print(f"  Total words    : {total_words}")
print(f"  TF('india')    : {tf_india:.4f}  → {sum(1 for t in exercise_doc if t=='india')}/{total_words}")
print(f"  TF('moon')     : {tf_moon:.4f}  → {sum(1 for t in exercise_doc if t=='moon')}/{total_words}")
print(f"  TF('rocket')   : {tf_rocket:.4f}  → word not found → 0/{total_words}")

# ✅ Expected: india=0.3333, moon=0.3333, rocket=0.0000

---
## 🔬 Part 2: Understanding Inverse Document Frequency (IDF)

### What is IDF?
IDF answers: **"How rare/special is this word across the ENTIRE collection?"**

```
IDF(word) = log( (N + 1) / (number of docs containing word + 1) ) + 1
```

Where:
- **N** = total number of documents
- The `+1` inside and outside is called **Laplace smoothing** to avoid division by zero
- The `log()` dampens extreme values — very rare words don't get astronomically high scores

**Intuition:**
- Word in 1/100 docs  → high IDF (rare, distinctive, important!)
- Word in 100/100 docs → low IDF (ubiquitous, not helpful for matching)

In [ ]:
# ── DEMO: Calculating IDF manually ──

def compute_idf(word, all_docs_tokenised, N):
    """
    Compute smoothed IDF of a word across a corpus.
    
    Parameters:
        word                 : str  — the word to score
        all_docs_tokenised   : list[list] — all documents as token lists
        N                    : int  — total number of documents
    
    Returns:
        float — IDF score (always >= 1.0 with smoothing)
    """
    # Count how many docs contain this word
    df = sum(1 for doc in all_docs_tokenised if word in doc)
    # Smoothed log IDF
    return math.log((N + 1) / (df + 1)) + 1

# Our corpus: 5 space-related documents
corpus = [
    "nasa launched the apollo mission to the moon",
    "isro achieved the cheapest successful moon mission chandrayaan",
    "esa operates many space projects across europe",
    "the moon has water ice at the poles discovered by chandrayaan",
    "spacex reusable rockets reduced launch costs significantly"
]
tokenised_corpus = [doc.split() for doc in corpus]
N = len(corpus)

# IDF for various words
test_words = ["the", "moon", "chandrayaan", "spacex", "mission", "launched"]

print("IDF Scores (N=5 documents):")
print(f"{'Word':<15} {'Docs Containing':>18} {'IDF Score':>12} {'Interpretation'}")
print("-" * 72)
for word in test_words:
    df = sum(1 for doc in tokenised_corpus if word in doc)
    idf = compute_idf(word, tokenised_corpus, N)
    interp = "very common" if idf < 1.3 else ("common" if idf < 1.7 else ("rare" if idf < 2.0 else "very rare / unique"))
    print(f"  {word:<13} {df:>18}   {idf:>10.4f}    {interp}")

print("\n📝 Words in MORE docs have LOWER IDF. Unique words have HIGHER IDF.")

In [ ]:
# ── Visualise: IDF vs Document Frequency ──

N_viz = 100  # imagine 100 documents
df_values = np.arange(1, 101)
idf_values = np.log((N_viz + 1) / (df_values + 1)) + 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df_values, idf_values, color='#4A90D9', linewidth=2.5)
ax.fill_between(df_values, idf_values, alpha=0.15, color='#4A90D9')

# Annotate key points
for df_pt, label in [(1, 'Very Rare\n(df=1)'), (10, 'Uncommon\n(df=10)'),
                     (50, 'Common\n(df=50)'), (100, 'Everywhere\n(df=100)')]:
    idf_pt = math.log((N_viz + 1) / (df_pt + 1)) + 1
    ax.scatter([df_pt], [idf_pt], s=80, color='#E74C3C', zorder=5)
    ax.annotate(f"{label}\nIDF={idf_pt:.2f}",
                xy=(df_pt, idf_pt), xytext=(df_pt + 3, idf_pt + 0.1),
                fontsize=8, color='#2C3E50')

ax.set_xlabel('Number of Documents Containing the Word (df)', fontsize=11)
ax.set_ylabel('IDF Score', fontsize=11)
ax.set_title('IDF Score vs Document Frequency\n(Higher IDF = rarer word = more distinctive)', fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('idf_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Chart saved as idf_curve.png")

### ✏️ Exercise 2.1 — Predict IDF ordering

**Before running the code**, predict which word will have the HIGHEST IDF in a 3-document corpus:
```
Doc1: "moon rocket launch pad"
Doc2: "rocket fuel combustion pad"
Doc3: "launch window mission control"
```
Candidates: `"rocket"`, `"pad"`, `"moon"`, `"launch"`

**Your prediction:** _____________

Then run the cell to verify:

In [ ]:
# ✏️ Exercise 2.1 — Verify your IDF prediction
ex_corpus = [
    "moon rocket launch pad".split(),
    "rocket fuel combustion pad".split(),
    "launch window mission control".split(),
]
ex_N = len(ex_corpus)
candidates = ["rocket", "pad", "moon", "launch"]

print("IDF scores for Exercise 2.1:")
idf_scores = {}
for word in candidates:
    df  = sum(1 for doc in ex_corpus if word in doc)
    idf = compute_idf(word, ex_corpus, ex_N)
    idf_scores[word] = idf
    print(f"  '{word}'  in {df} doc(s) → IDF = {idf:.4f}")

winner = max(idf_scores, key=idf_scores.get)
print(f"\n✅ Highest IDF: '{winner}' = {idf_scores[winner]:.4f}")
print("   (Words appearing in ONLY 1 doc get the highest IDF)")

# Visualise
fig, ax = plt.subplots(figsize=(6, 3))
words_sorted = sorted(idf_scores, key=idf_scores.get, reverse=True)
colors = ['#E74C3C' if w == winner else '#4A90D9' for w in words_sorted]
ax.bar(words_sorted, [idf_scores[w] for w in words_sorted], color=colors, alpha=0.85)
ax.set_ylabel('IDF Score')
ax.set_title('Exercise 2.1 — IDF Scores')
for i, w in enumerate(words_sorted):
    ax.text(i, idf_scores[w] + 0.01, f"{idf_scores[w]:.3f}", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 🔬 Part 3: Combining TF × IDF — The Full Score

### The Final Formula
```
TF-IDF(word, doc) = TF(word, doc) × IDF(word)
```

This rewards words that are:
1. **Frequent in this document** (high TF)
2. **Rare across the collection** (high IDF)

**The Space Agency Example from the RAG article:**
- Query: `"space agency with a successful yet cheapest moon mission"`
- Doc1: NASA/Apollo
- Doc2: ISRO/Chandrayaan  ← should win because it has "cheapest" and "successful"
- Doc3: ESA/Europe

In [ ]:
# ── DEMO: Full TF-IDF scoring (exact example from the RAG article) ──

documents = [
    "nasa launched the apollo mission to the moon",         # Doc 1
    "isro achieved the cheapest successful moon mission chandrayaan",  # Doc 2  
    "esa operates many space projects across europe"         # Doc 3
]
query = "space agency with a successful yet cheapest moon mission"

tokenised_docs = [doc.split() for doc in documents]
N = len(tokenised_docs)
query_words = ["space", "agency", "successful", "cheapest", "moon", "mission"]

# Build a detailed scoring table
rows = []
doc_scores = [0.0, 0.0, 0.0]

for word in query_words:
    idf  = compute_idf(word, tokenised_docs, N)
    tfs  = [compute_tf(word, doc) for doc in tokenised_docs]
    tfidf_scores = [tf * idf for tf in tfs]
    for i, s in enumerate(tfidf_scores):
        doc_scores[i] += s
    rows.append({
        "Word":   word,
        "IDF":    round(idf, 4),
        "TF-D1":  round(tfs[0], 4),   "Score-D1": round(tfidf_scores[0], 4),
        "TF-D2":  round(tfs[1], 4),   "Score-D2": round(tfidf_scores[1], 4),
        "TF-D3":  round(tfs[2], 4),   "Score-D3": round(tfidf_scores[2], 4),
    })

df_scores = pd.DataFrame(rows)
print("TF-IDF Score Breakdown:")
print(df_scores.to_string(index=False))
print(f"\nTOTAL: Doc1={doc_scores[0]:.4f}  Doc2={doc_scores[1]:.4f}  Doc3={doc_scores[2]:.4f}")

ranked = sorted(enumerate(doc_scores), key=lambda x: -x[1])
print("\n🏆 Final Ranking:")
for rank, (idx, score) in enumerate(ranked, 1):
    print(f"  Rank {rank}: Doc {idx+1} (score={score:.4f}) → {documents[idx]}")

In [ ]:
# ── Heatmap: TF-IDF scores per word per document ──

heatmap_data = np.zeros((len(query_words), len(documents)))
for i, word in enumerate(query_words):
    idf = compute_idf(word, tokenised_docs, N)
    for j, doc in enumerate(tokenised_docs):
        heatmap_data[i, j] = compute_tf(word, doc) * idf

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_data, 
            annot=True, fmt='.3f', 
            xticklabels=[f'Doc {i+1}' for i in range(len(documents))],
            yticklabels=query_words,
            cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('TF-IDF Score Heatmap\n(darker = higher score = more relevant word-doc pair)', fontsize=11)
ax.set_xlabel('Documents')
ax.set_ylabel('Query Words')
plt.tight_layout()
plt.savefig('tfidf_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Saved: tfidf_heatmap.png")

---
## 🔬 Part 4: Production TF-IDF with scikit-learn

In [ ]:
# ── Using TfidfVectorizer (industry standard) ──

# A larger, more realistic knowledge base
knowledge_base = [
    "NASA launched Apollo 11 and sent the first humans to the Moon in 1969.",
    "ISRO's Chandrayaan mission proved the Moon has water ice at the poles.",
    "SpaceX developed reusable Falcon 9 rockets to dramatically reduce launch costs.",
    "The Hubble Space Telescope orbits Earth and captures deep-space images.",
    "Mars rovers like Curiosity and Perseverance have been exploring Mars since 2012.",
    "ESA's Rosetta probe successfully landed on a comet for the first time in history.",
    "The James Webb Space Telescope replaced Hubble with infrared imaging capability.",
    "SpaceX Starship aims to carry humans to Mars within this decade.",
]

# Step 1: Fit vectorizer on the knowledge base
vectorizer = TfidfVectorizer(
    stop_words='english',    # Remove common words like 'the', 'a', 'is'
    lowercase=True,          # Normalise case
    ngram_range=(1, 2),      # Include both single words and 2-word phrases
    max_features=50          # Limit vocabulary size
)
doc_matrix = vectorizer.fit_transform(knowledge_base)

print(f"Knowledge base: {len(knowledge_base)} documents")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)} unique terms")
print(f"Document matrix shape: {doc_matrix.shape} (docs × terms)")
print(f"\nSample vocabulary terms: {list(vectorizer.vocabulary_.keys())[:15]}")

In [ ]:
# ── Querying the TF-IDF index ──

def tfidf_search(query_text, vectorizer, doc_matrix, documents, top_k=3):
    """
    Search a TF-IDF index for relevant documents.
    
    Parameters:
        query_text  : str   — the search query
        vectorizer  : fitted TfidfVectorizer
        doc_matrix  : sparse matrix of document TF-IDF vectors
        documents   : list[str] — original documents
        top_k       : int   — number of results to return
    
    Returns:
        list of (score, document) tuples
    """
    query_vec = vectorizer.transform([query_text])
    scores    = cosine_similarity(query_vec, doc_matrix).flatten()
    top_idx   = np.argsort(-scores)[:top_k]
    return [(scores[i], documents[i]) for i in top_idx]

# Run several test queries
test_queries = [
    "cheapest rocket launch",
    "humans landing on Mars",
    "telescope images space",
    "Moon water discovery",
]

for q in test_queries:
    print(f"\n🔍 Query: {q!r}")
    results = tfidf_search(q, vectorizer, doc_matrix, knowledge_base, top_k=2)
    for i, (score, doc) in enumerate(results, 1):
        print(f"  {i}. (score={score:.4f}) {doc}")

### ✏️ Exercise 4.1 — Add to the knowledge base and query

Add 3 new documents about your own topic of interest, rebuild the index, and run a custom query.

In [ ]:
# ✏️ YOUR TURN — Add your own documents and query!

# TODO: Add 3 documents about any topic you like
my_docs = knowledge_base + [
    "TODO: Add your first document here.",
    "TODO: Add your second document here.",
    "TODO: Add your third document here.",
]

# Rebuild the index with new documents
new_vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
new_matrix     = new_vectorizer.fit_transform(my_docs)

# TODO: Write a query that targets your new documents
my_query = "TODO: Write your query here"

results = tfidf_search(my_query, new_vectorizer, new_matrix, my_docs, top_k=3)
print(f"Query: {my_query!r}\n")
for i, (score, doc) in enumerate(results, 1):
    print(f"  {i}. (score={score:.4f}) {doc}")

---
## 📋 Summary & Key Takeaways

| Concept | Formula | What it measures |
|---------|---------|------------------|
| **TF** | count(w,d) / len(d) | Word importance *within* one document |
| **IDF** | log((N+1)/(df+1)) + 1 | Word *rarity* across the entire collection |
| **TF-IDF** | TF × IDF | Combined importance: frequent-in-doc + rare-across-collection |

### ⚠️ Limitations of TF-IDF
- **Exact match only**: `"car"` and `"automobile"` are treated as completely different words
- **No word order**: `"dog bites man"` and `"man bites dog"` get the same score
- **No meaning**: TF-IDF doesn't understand context or semantics

→ This is why **semantic search** (embeddings) was developed. See Exercise 03!

### ✅ When to use TF-IDF?
- Fast keyword retrieval on large corpora
- When exact term matching matters (legal documents, code search)
- As the keyword component in **hybrid search**
- Quick baseline before implementing more complex semantic search

In [ ]:
# ── Final Summary: Print all key formulas and outputs ──
print("="*60)
print("EXERCISE 01 — TF-IDF: COMPLETE REFERENCE")
print("="*60)
print()
print("FORMULAS:")
print("  TF(w, d)      = count(w in d) / |d|")
print("  IDF(w)        = log((N+1) / (df(w)+1)) + 1")
print("  TF-IDF(w, d)  = TF(w,d) × IDF(w)")
print()
print("KEY INSIGHTS:")
print("  • Common words (the, is, a)  → LOW IDF  → LOW TF-IDF")
print("  • Rare, unique words         → HIGH IDF → HIGH TF-IDF")
print("  • Words absent from doc      → TF=0    → TF-IDF=0")
print()
print("STRENGTHS:            | WEAKNESSES:")
print("  Fast, interpretable  |   No semantic understanding")
print("  Exact term matching  |   No synonym handling")
print("  Production-proven    |   Order-insensitive")
print("="*60)